In [45]:
import math
import random

class Value:
    def __init__(self, data, child=(), op='', label=''):
        self.data = data
        self.prev = set(child)
        self.op = op
        self.grad = 0.0
        self._backward = lambda: None
        self.label = label

    def backward(self):
        topo = []
        visited = set()
        def build_topo(node):
            if node not in visited:
                visited.add(node)
                for child in node.prev:
                    build_topo(child)
                topo.append(node)
        
        build_topo(self)

        self.grad = 1.0
        for val in reversed(topo):
            val._backward()
    
    def __repr__(self):
        return f"Value(data = {self.data})"
    
    def __add__(self, other):
        other = other if isinstance(other, Value) else Value(other)
        out = Value(self.data + other.data, (self, other), "+")
        
        def _backward():
            self.grad += out.grad
            other.grad += out.grad
        out._backward = _backward
        return out
    
    def __radd__(self, other): # other + self
        return self + other
    
    def __mul__(self, other):
        other = other if isinstance(other, Value) else Value(other)
        out = Value(self.data * other.data, (self, other), '*')

        def _backward():
            self.grad += other.data * out.grad
            other.grad += self.data * out.grad
        out._backward = _backward

        return out
    
    def __neg__(self): # -self
        return self * -1

    def __sub__(self, other): # self - other
        return self + -other
    
    def __pow__(self, other): # self ** other
        assert isinstance(other, (int, float))
        out = Value(self.data ** other, (self,), f"**{other}")

        def _backward():
            self.grad += other * self.data**(other-1) * out.grad
        out._backward = _backward

        return out
    
    def __rmul__(self, other): # other * self
        return self * other
    
    def __truediv__(self, other): # self / other
        return self * other ** -1

    def tanh(self):
        x = self.data
        t = (math.exp(x * 2) - 1) / (math.exp(x * 2) + 1)
        out = Value(t, (self, ), label='tanh')

        def _backward():
            self.grad += (1 - t**2) * out.grad

        out._backward = _backward
        return out
    
    def relu(self):
        out = Value(max(0, self.data), (self, ), label='relu')

        def _backward():
            self.grad += (out.data > 0) * out.grad
            
        out._backward = _backward
        return out
    
    def exp(self):
        out = Value(math.exp(self.data), (self, ), label='exp')

        def _backward():
            self.grad += out.data * out.grad

        out._backward = _backward
        return out

In [46]:
# inputs x1,x2
x1 = Value(2.0, label='x1')
x2 = Value(0.0, label='x2')
# weights w1,w2
w1 = Value(-3.0, label='w1')
w2 = Value(1.0, label='w2')
# bias of the neuron
b = Value(6.8813735870195432, label='b')
# x1*w1 + x2*w2 + b
x1w1 = x1*w1; x1w1.label = 'x1*w1'
x2w2 = x2*w2; x2w2.label = 'x2*w2'
x1w1x2w2 = x1w1 + x2w2; x1w1x2w2.label = 'x1*w1 + x2*w2'
n = x1w1x2w2 + b; n.label = 'n'
o = n.tanh(); o.label = 'o'
o.backward()
x1.grad

-1.4999999999999996

In [47]:
# inputs x1,x2
x1 = Value(2.0, label='x1')
x2 = Value(0.0, label='x2')
# weights w1,w2
w1 = Value(-3.0, label='w1')
w2 = Value(1.0, label='w2')
# bias of the neuron
b = Value(6.8813735870195432, label='b')
# x1*w1 + x2*w2 + b
x1w1 = x1*w1; x1w1.label = 'x1*w1'
x2w2 = x2*w2; x2w2.label = 'x2*w2'
x1w1x2w2 = x1w1 + x2w2; x1w1x2w2.label = 'x1*w1 + x2*w2'
n = x1w1x2w2 + b; n.label = 'n'
# ----
e = (2*n).exp() 
o = (e - 1) / (e + 1)
o.label = 'o'
# ----
o.backward()
x1.grad

-1.5

In [ ]:
class Neuron:
    def __init__(self, nin):
        self.w = [(Value(random.uniform(-1, 1))) for _ in range(nin)]
        self.b = Value(random.uniform(-1, 1))
    
    def __call__(self, x):
        # wi * xi + b
        # print(list(zip(self.w, x)))
        act = sum((wi*xi for wi, xi in zip(self.w, x)), self.b)
        out = act.tanh()
        return out
    
    def parameters(self):
        return self.w + [self.b]

class Layer:
    def __init__(self, nin, nout):
        self.neurons = [Neuron(nin) for _ in range(nout)]

    def __call__(self, x):
        outs = [n(x) for n in self.neurons]
        return outs[0] if len(outs) == 1 else outs

    def parameters(self):
        return [p for n in self.neurons for p in n.parameters()]
        # ps = []
        # for n in self.neurons:
        #     ps.extend(n.parameters())
        # return ps

class MLP:
    def __init__(self, nin, nouts):
        sz = [nin] + nouts
        self.layers = [Layer(sz[i], sz[i+1]) for i in range(len(nouts))]

    def __call__(self, x):
        for layer in self.layers:
            x = layer(x)
        return x
    
    def parameters(self):
        return [p for l in self.layers for p in l.parameters()]

In [49]:
# x = [2.0, 3.0]
# n = Neuron(2)
# n(x)

n = MLP(3, [4, 4, 1])

In [50]:
xs = [
    [2.0, 3.0, -1.0],
    [3.0, -1.0, 0.5],
    [0.5, 1.0, 1.0],
    [1.0, 1.0, -1.0],
]
ys = [1.0, -1.0, -1.0, 1.0] # desired targets
ypred = [n(x) for x in xs]
ypred

[Value(data = 0.38561325277027625),
 Value(data = 0),
 Value(data = 0),
 Value(data = 0.38561325277027625)]

In [55]:
for k in range(100):
    # forward pass
    ypred = [n(x) for x in xs]
    loss = sum((yout - ygt)**2 for ygt, yout in zip(ys, ypred))

    # backward pass
    for p in n.parameters():
        p.grad = 0
    loss.backward()

    for p in n.parameters():
        p.data += -0.01 * p.grad

    print(k, loss)

0 Value(data = 2.0000000007079217)
1 Value(data = 2.000000000652421)
2 Value(data = 2.000000000601271)
3 Value(data = 2.0000000005541314)
4 Value(data = 2.0000000005106875)
5 Value(data = 2.0000000004706493)
6 Value(data = 2.000000000433751)
7 Value(data = 2.0000000003997442)
8 Value(data = 2.000000000368405)
9 Value(data = 2.0000000003395217)
10 Value(data = 2.0000000003129035)
11 Value(data = 2.000000000288372)
12 Value(data = 2.0000000002657634)
13 Value(data = 2.0000000002449276)
14 Value(data = 2.000000000225725)
15 Value(data = 2.0000000002080283)
16 Value(data = 2.0000000001917186)
17 Value(data = 2.000000000176688)
18 Value(data = 2.000000000162836)
19 Value(data = 2.0000000001500693)
20 Value(data = 2.0000000001383036)
21 Value(data = 2.0000000001274607)
22 Value(data = 2.000000000117468)
23 Value(data = 2.0000000001082583)
24 Value(data = 2.000000000099771)
25 Value(data = 2.000000000091949)
26 Value(data = 2.00000000008474)
27 Value(data = 2.0000000000780966)
28 Value(data =

In [56]:
ypred

[Value(data = 0.9999996693795149),
 Value(data = 0),
 Value(data = 0),
 Value(data = 0.9999996693795149)]